# AMR-UnderDifferentNoises-DL: Sonuç Analizi ve Karşılaştırma

Bu notebook, fine-tuning işlemi sonrasında elde edilen `.pkl` uzantılı sonuç (accuracy, f1, mcc vb.) dosyalarını Google Drive'dan okur ve **modeller arası ile veri setleri arası kıyaslamaları** tek bir grafik üzerinde birleştirir.

Öğrenilecekler:
- Hangi model daha başarılı? (MCLDNN vs PETCGDNN)
- Hangi sönümleme (fading) kanalı daha yıkıcı? (Rayleigh vs Rician K3 vs Rician K10)

In [ ]:
# 1. Drive'a Bağlan ve Gerekli Kütüphaneleri Yükle
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Kayıtların bulunduğu klasör (Eğer farklıysa burayı güncelleyin)
RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/fine_tuning_results'

if not os.path.exists(RESULTS_DIR):
    print("Uyarı: fine_tuning_results klasörü bulunamadı. Alternatif klasöre bakılıyor...")
    RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/results'

print(f"\nSonuçların okunacağı dizin: {RESULTS_DIR}")


In [ ]:
# 2. Verileri Okuma Fonksiyonu
def load_pkl_data(model, dataset, metric_file='acc.pkl'):
    folder_name = f"{model}_{dataset}"
    file_path = os.path.join(RESULTS_DIR, folder_name, 'predictions', metric_file)
    
    # prediction klasörü yoksa direkt klasör içine bak
    if not os.path.exists(file_path):
        file_path = os.path.join(RESULTS_DIR, folder_name, metric_file)
        
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            return pickle.load(f)
    else:
        print(f"HATA: Dosya bulunamadı -> {file_path}")
        return None

models = ['mcldnn', 'petcgdnn']
datasets = ['rayleigh', 'rician_k3', 'rician_k10']


In [ ]:
# 3. Model Karşılaştırması (Aynı Veri Setinde MCLDNN vs PETCGDNN)
def plot_model_comparison(dataset_name, dataset_title):
    mcldnn_acc = load_pkl_data('mcldnn', dataset_name, 'acc.pkl')
    petcgdnn_acc = load_pkl_data('petcgdnn', dataset_name, 'acc.pkl')
    
    if mcldnn_acc is None or petcgdnn_acc is None:
        return
        
    snrs_sorted = sorted(list(mcldnn_acc.keys()))
    mcl_vals = [mcldnn_acc[s]*100 for s in snrs_sorted]
    pet_vals = [petcgdnn_acc[s]*100 for s in snrs_sorted]
    
    plt.figure(figsize=(10, 6), dpi=120)
    plt.plot(snrs_sorted, mcl_vals, 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    plt.plot(snrs_sorted, pet_vals, 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    
    plt.xlabel("Signal to Noise Ratio (SNR) in dB", fontsize=12)
    plt.ylabel("Classification Accuracy (%)", fontsize=12)
    plt.title(f"Model Comparison on {dataset_title} Channel", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 105])
    plt.tight_layout()
    plt.show()

print("========== MODEL KARŞILAŞTIRMALARI (MCLDNN vs PETCGDNN) ==========")
plot_model_comparison('rayleigh', 'Rayleigh Fading')
plot_model_comparison('rician_k3', 'Rician Fading (K=3)')
plot_model_comparison('rician_k10', 'Rician Fading (K=10)')


In [ ]:
# 4. Kanal (Gürültü Tipi) Karşılaştırması (Aynı Modelde Rayleigh vs Rician K3 vs K10)
def plot_channel_comparison(model_name, model_title):
    rayleigh_acc = load_pkl_data(model_name, 'rayleigh', 'acc.pkl')
    rician3_acc = load_pkl_data(model_name, 'rician_k3', 'acc.pkl')
    rician10_acc = load_pkl_data(model_name, 'rician_k10', 'acc.pkl')
    
    if rayleigh_acc is None or rician3_acc is None or rician10_acc is None:
        return
        
    snrs_sorted = sorted(list(rayleigh_acc.keys()))
    ray_vals = [rayleigh_acc[s]*100 for s in snrs_sorted]
    ric3_vals = [rician3_acc[s]*100 for s in snrs_sorted]
    ric10_vals = [rician10_acc[s]*100 for s in snrs_sorted]
    
    plt.figure(figsize=(10, 6), dpi=120)
    plt.plot(snrs_sorted, ray_vals, 'o-', linewidth=2.5, label='Rayleigh', color='#E63946')
    plt.plot(snrs_sorted, ric3_vals, 's-', linewidth=2.5, label='Rician K=3', color='#2A9D8F')
    plt.plot(snrs_sorted, ric10_vals, '^-', linewidth=2.5, label='Rician K=10', color='#457B9D')
    
    plt.xlabel("Signal to Noise Ratio (SNR) in dB", fontsize=12)
    plt.ylabel("Classification Accuracy (%)", fontsize=12)
    plt.title(f"Channel Fading Impact on {model_title} Model", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 105])
    plt.tight_layout()
    plt.show()

print("========== KANAL KARŞILAŞTIRMALARI (Rayleigh vs Rician) ==========")
plot_channel_comparison('mcldnn', 'MCLDNN')
plot_channel_comparison('petcgdnn', 'PET-CGDNN')


In [ ]:
# 5. İstatistiksel Karşılaştırma Raporu (Yazılı Çıktı)
print("========== YAZILI PERFORMANS ÖZET RAPORU ==========\n")

for dataset in datasets:
    print(f"[{dataset.upper()}] İÇİN DOĞRULUK ORANLARI:")
    for model in models:
        acc_data = load_pkl_data(model, dataset, 'acc.pkl')
        if acc_data:
            avg_acc = np.mean(list(acc_data.values())) * 100
            max_acc = np.max(list(acc_data.values())) * 100
            print(f"  - {model.upper():<8} : Ortalama = %{avg_acc:.2f} | En Yüksek SNR Başarısı = %{max_acc:.2f}")
    print("-" * 50)
